In [1]:
"""
Sign Language Detection — Transformer Model
==========================================
Input  : (batch, 32, 1040)  — 32 frames × 1040 landmark features
Output : (batch, 2000)       — class logits
"""

import os, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path
from tqdm import tqdm

# ─────────────────────────────────────────────
#  Config
# ─────────────────────────────────────────────
CFG = dict(
    # paths
    npy_root    = "/kaggle/input/datasets/mohamedmedhaffar/preprocessed-data/npy/npy",
    label_json  = "/kaggle/input/datasets/mohamedmedhaffar/preprocessed-data/nslt_2000.json",
    class_list  = "/kaggle/input/datasets/mohamedmedhaffar/preprocessed-data/wlasl_class_list.txt",

    # data
    num_frames  = 32,
    num_joints  = 520,       # landmarks per axis
    num_axes    = 2,         # x, y
    feat_dim    = 1040,      # 520 * 2  (flattened per frame)
    num_classes = 2000,

    # model
    d_model     = 256,       # transformer hidden dim
    nhead       = 8,
    num_layers  = 4,
    dim_ff      = 512,
    dropout     = 0.1,

    # training
    epochs      = 60,
    batch_size  = 32,
    lr          = 3e-4,
    weight_decay= 1e-4,
    grad_clip   = 1.0,
    label_smooth= 0.1,

    device      = "cuda" if torch.cuda.is_available() else "cpu",
    num_workers = 4,
    seed        = 42,
)


# ─────────────────────────────────────────────
#  Dataset
# ─────────────────────────────────────────────
class SignLanguageDataset(Dataset):
    """
    Each sample:
      x  : Tensor (32, 1040)  — frames × flattened landmarks
      y  : int                — class index (0-1999)
    """

    def __init__(self, label_json: str, npy_root: str, subset: str, augment: bool = False):
        super().__init__()
        self.npy_root = Path(npy_root)
        self.augment  = augment
        self.num_frames = CFG["num_frames"]

        with open(label_json) as f:
            meta = json.load(f)

        self.samples = [
            (vid_id, info["action"][0])
            for vid_id, info in meta.items()
            if info["subset"] == subset
        ]

    def __len__(self):
        return len(self.samples)

    def _load_video(self, vid_id: str) -> np.ndarray:
        """Load all 32 frames for a video. Returns (32, 1040)."""
        vid_dir = self.npy_root / vid_id
        frames  = []

        for i in range(self.num_frames):
            frame_path = vid_dir / f"frame_{i:03d}.npy"
            if frame_path.exists():
                arr = np.load(frame_path)   # (520, 2)  — rows=landmarks, cols=axes
                # flatten to (1040,): [x0,y0, x1,y1, ...] interleaved order
                flat = arr.flatten(order="C")  # all x then all y → (1040,)
            else:
                print("frames are missing")
                flat = np.zeros(CFG["feat_dim"], dtype=np.float32)

            frames.append(flat)

        return np.stack(frames, axis=0).astype(np.float32)  # (32, 1040)

    def _augment(self, x: np.ndarray) -> np.ndarray:
        """Lightweight temporal + spatial augmentation."""
        # 1. Temporal jitter: random roll along time axis
        shift = np.random.randint(-4, 5)
        x = np.roll(x, shift, axis=0)

        # 2. Scale jitter on coordinates
        scale = np.random.uniform(0.9, 1.1)
        x = x * scale

        # 3. Gaussian noise
        x = x + np.random.normal(0, 0.005, x.shape).astype(np.float32)

        return x

    def __getitem__(self, idx):
        vid_id, label = self.samples[idx]
        x = self._load_video(vid_id)

        if self.augment:
            x = self._augment(x)

        return torch.from_numpy(x), torch.tensor(label, dtype=torch.long)


# ─────────────────────────────────────────────
#  Model
# ─────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""

    def __init__(self, d_model: int, max_len: int = 32, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, T, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


class SignTransformer(nn.Module):
    """
    Transformer encoder for sign language classification.

    Architecture:
      Input projection  : feat_dim (1040) → d_model
      Positional Encoding
      N × TransformerEncoderLayer
      Aggregation       : mean-pool over time  (robust vs CLS token for short seqs)
      Classifier head   : d_model → num_classes
    """

    def __init__(self, cfg: dict):
        super().__init__()
        d_model    = cfg["d_model"]
        nhead      = cfg["nhead"]
        num_layers = cfg["num_layers"]
        dim_ff     = cfg["dim_ff"]
        dropout    = cfg["dropout"]
        feat_dim   = cfg["feat_dim"]
        num_cls    = cfg["num_classes"]

        # — Input projection —
        self.input_proj = nn.Sequential(
            nn.Linear(feat_dim, d_model),
            nn.LayerNorm(d_model),
        )

        # — Positional encoding —
        self.pos_enc = PositionalEncoding(d_model, max_len=cfg["num_frames"], dropout=dropout)

        # — Transformer encoder —
        enc_layer = nn.TransformerEncoderLayer(
            d_model      = d_model,
            nhead        = nhead,
            dim_feedforward = dim_ff,
            dropout      = dropout,
            batch_first  = True,   # (B, T, d_model)
            norm_first   = True,   # Pre-LN: more stable for deep nets
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                             norm=nn.LayerNorm(d_model))

        # — Classifier head —
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_cls),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, feat_dim)  — T=32, feat_dim=1040
        Returns:
            logits: (B, num_classes)
        """
        x = self.input_proj(x)    # (B, T, d_model)
        x = self.pos_enc(x)       # + positional encoding
        x = self.encoder(x)       # (B, T, d_model)
        x = x.mean(dim=1)         # mean-pool → (B, d_model)
        return self.head(x)       # (B, num_classes)


# ─────────────────────────────────────────────
#  Label-smoothed Cross-Entropy
# ─────────────────────────────────────────────
class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes: int, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        confidence = 1.0 - self.smoothing
        log_probs  = F.log_softmax(logits, dim=-1)
        nll        = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_probs.mean(dim=-1)
        return (confidence * nll + self.smoothing * smooth_loss).mean()


# ─────────────────────────────────────────────
#  Training helpers
# ─────────────────────────────────────────────
def top_k_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 5) -> float:
    _, pred = logits.topk(k, dim=1)
    correct = pred.eq(targets.view(-1, 1).expand_as(pred))
    return correct.any(dim=1).float().mean().item()


def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip):
    model.train()
    total_loss, total_acc1, total_acc5, n = 0, 0, 0, 0

    for x, y in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc1 += (logits.argmax(1) == y).float().sum().item()
        total_acc5 += top_k_accuracy(logits, y, k=5) * bs
        n += bs

    return total_loss / n, total_acc1 / n, total_acc5 / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc1, total_acc5, n = 0, 0, 0, 0

    for x, y in tqdm(loader, desc="Val  ", leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits, y)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc1 += (logits.argmax(1) == y).float().sum().item()
        total_acc5 += top_k_accuracy(logits, y, k=5) * bs
        n += bs

    return total_loss / n, total_acc1 / n, total_acc5 / n


# ─────────────────────────────────────────────
#  Main
# ─────────────────────────────────────────────
def main():
    torch.manual_seed(CFG["seed"])
    device = CFG["device"]
    print(f"Device: {device}")

    # ── Datasets ──────────────────────────────
    train_ds = SignLanguageDataset(CFG["label_json"], CFG["npy_root"], "train", augment=True)
    val_ds   = SignLanguageDataset(CFG["label_json"], CFG["npy_root"], "val",   augment=False)
    test_ds  = SignLanguageDataset(CFG["label_json"], CFG["npy_root"], "test",  augment=False)

    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}")

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                              num_workers=CFG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                              num_workers=CFG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,
                              num_workers=CFG["num_workers"], pin_memory=True)

    # ── Model ─────────────────────────────────
    model = SignTransformer(CFG).to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {total_params:,}")

    # ── Loss / Optimizer / Scheduler ──────────
    criterion = LabelSmoothingLoss(CFG["num_classes"], CFG["label_smooth"])
    optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-6)

    # ── Training loop ─────────────────────────
    best_val_acc = 0.0

    for epoch in range(1, CFG["epochs"] + 1):
        train_loss, train_acc1, train_acc5 = train_one_epoch(
            model, train_loader, optimizer, criterion, device, CFG["grad_clip"])
        val_loss, val_acc1, val_acc5 = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        improved = val_acc1 > best_val_acc
        if improved:
            best_val_acc = val_acc1
            torch.save({"epoch": epoch, "model": model.state_dict(),
                        "val_acc1": val_acc1}, "best_model.pt")

        marker = " ★" if improved else ""
        print(
            f"Epoch {epoch:3d}/{CFG['epochs']}  "
            f"| Train  loss={train_loss:.4f}  acc@1={train_acc1:.3f}  acc@5={train_acc5:.3f}  "
            f"| Val    loss={val_loss:.4f}  acc@1={val_acc1:.3f}  acc@5={val_acc5:.3f}"
            f"{marker}"
        )

    # ── Final test evaluation ─────────────────
    ckpt = torch.load("best_model.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    test_loss, test_acc1, test_acc5 = evaluate(model, test_loader, criterion, device)
    print(f"\n[TEST]  loss={test_loss:.4f}  acc@1={test_acc1:.3f}  acc@5={test_acc5:.3f}")


if __name__ == "__main__":
    main()

Device: cuda
Train: 14,296  |  Val: 3,920  |  Test: 2,879


/tmp/ipykernel_57/1860202666.py:189: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers,


Parameters: 2,666,832


Epoch   1/60  | Train  loss=7.6075  acc@1=0.000  acc@5=0.002  | Val    loss=7.6028  acc@1=0.001  acc@5=0.003 ★


Epoch   2/60  | Train  loss=7.5927  acc@1=0.002  acc@5=0.006  | Val    loss=7.5917  acc@1=0.002  acc@5=0.006 ★


...

Epoch  24/60  | Train  loss=6.6058  acc@1=0.016  acc@5=0.062  | Val    loss=7.7331  acc@1=0.003  acc@5=0.014


Epoch  25/60  | Train  loss=6.5683  acc@1=0.020  acc@5=0.068  | Val    loss=7.7573  acc@1=0.004  acc@5=0.017


Epoch  26/60  | Train  loss=6.5316  acc@1=0.019  acc@5=0.074  | Val    loss=7.7827  acc@1=0.004  acc@5=0.015


Epoch  27/60  | Train  loss=6.4936  acc@1=0.023  acc@5=0.081  | Val    loss=7.8004  acc@1=0.004  acc@5=0.016 ★


KeyboardInterrupt: 